# Lesson 5 · Asking Business Questions: Grouping, Aggregating, Pivoting & Merging
### From rows of records to answers that a decision-maker can use

| | |
|---|---|
| **Type** | SOLUTIONS 05 |
| **Time** | about 150 minutes |
| **Level** | Intermediate |
| **Prerequisites** | Lessons 3–4 |

**By the end of this notebook you will be able to:**

- Explain *split → apply → combine* and use `groupby` with `agg`, named aggregation, `transform` and `filter`
- Build league tables, volatility measures and within-group comparisons
- Reshape with `pivot_table`, `crosstab` and `melt`
- Combine tables with `concat` and `merge` (inner/left/right/outer), and guard against silent row explosions and key mismatches
- Assemble a multi-source dataset (prices + production + weather + counties) ready for analysis

## 🛠️ Setup (identical to the lesson notebook)

In [1]:

import sys; sys.path.insert(0, "..")
import json
import numpy as np
import pandas as pd
from course_utils import DATA_DIR, check, first_look

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

prices = pd.read_csv(DATA_DIR / "market_prices_clean.csv", parse_dates=["date"])
prices["year"] = prices["date"].dt.year
prices["month"] = prices["date"].dt.month

production = pd.read_excel(DATA_DIR / "production_by_county.xlsx", sheet_name="production", skiprows=2)
counties = pd.read_excel(DATA_DIR / "production_by_county.xlsx", sheet_name="counties")
weather = pd.read_json(DATA_DIR / "weather_monthly.json")
with open(DATA_DIR / "markets.json") as f:
    markets = pd.json_normalize(json.load(f))
markets.columns = markets.columns.str.replace(".", "_", regex=False)
print("Setup complete ✔  prices", prices.shape, "| production", production.shape, "| weather", weather.shape)


Setup complete ✔  prices (15048, 11) | production (204, 5) | weather (432, 5)


### 🧩 Problem 5.1: Mean price by crop

Compute the **mean price per crop** and return it as a dictionary `{crop: mean}` with values rounded to 2 decimals.

*Expected type:* dict.

In [2]:
answer = prices.groupby("crop")["price_kes_per_kg"].mean().round(2).to_dict()

check('05-p1', answer)

✅ 05-p1: correct! Nicely done.


True

### 🧩 Problem 5.2: Tomato capital

Which **county** has the highest **mean tomato price**?

*Expected type:* a string.

In [3]:
answer = prices[prices["crop"] == "Tomatoes"].groupby("county")["price_kes_per_kg"].mean().idxmax()

check('05-p2', answer)

✅ 05-p2: correct! Nicely done.


True

### 🧩 Problem 5.3: Maize in 2022 (pivot)

Build a pivot table of **mean price** with crops as rows and years as columns. Return the cell for **Maize, 2022**, rounded to 2 decimals.

*Expected type:* float.

In [4]:
pt = prices.pivot_table(index="crop", columns="year", values="price_kes_per_kg", aggfunc="mean")
answer = round(float(pt.loc["Maize", 2022]), 2)

check('05-p3', answer)

✅ 05-p3: correct! Nicely done.


True

### 🧩 Problem 5.4: The biggest spike (transform)

Within **each (crop, market) series**, standardise the price: `(price − series mean) / series std` (pandas default `std`). Return the **largest** standardised value in the whole table, rounded to 2 decimals.

*Expected type:* float.

In [5]:
z = prices.groupby(["crop", "market"])["price_kes_per_kg"].transform(lambda s: (s - s.mean()) / s.std())
answer = round(float(z.max()), 2)

check('05-p4', answer)

✅ 05-p4: correct! Nicely done.


True

### 🧩 Problem 5.5: Most volatile crop

Using the **pooled** price of each crop (all markets, all weeks), compute the coefficient of variation (`std / mean`). Which crop has the **highest** CV?

*Expected type:* a string.

In [6]:
g = prices.groupby("crop")["price_kes_per_kg"]
answer = (g.std() / g.mean()).idxmax()

check('05-p5', answer)

✅ 05-p5: correct! Nicely done.


True

### 🧩 Problem 5.6: Top maize region in 2023 (merge)

Merge `production` with `counties` (left join on `county`). Among **Maize** rows in **2023**, which **region** had the highest total `production_tonnes`?

*Expected type:* a string.

In [7]:
m = production.merge(counties, on="county", how="left")
m = m[(m["crop"] == "Maize") & (m["year"] == 2023)]
answer = m.groupby("region")["production_tonnes"].sum().idxmax()

check('05-p6', answer)

✅ 05-p6: correct! Nicely done.


True

### 🧩 Problem 5.7: Orphan rows

How many rows of `production` (as loaded in Setup, *unfixed*) have a `county` that does **not** exist in the `counties` table?

*Expected type:* integer.

In [8]:
answer = int((~production["county"].isin(counties["county"])).sum())

check('05-p7', answer)

✅ 05-p7: correct! Nicely done.


True

### 🧩 Problem 5.8: Does rain move tomato prices?

Compute the **monthly mean tomato price per county** (group by `county, year, month`). Merge it with `weather` on `['county','year','month']` (inner join). Return the **Pearson correlation** between the mean price and `rainfall_mm`, rounded to 3 decimals.

*Expected type:* float.

In [9]:
t = prices[prices["crop"] == "Tomatoes"]
mt = t.groupby(["county", "year", "month"], as_index=False)["price_kes_per_kg"].mean()
m = mt.merge(weather, on=["county", "year", "month"], how="inner")
answer = round(float(m["price_kes_per_kg"].corr(m["rainfall_mm"])), 3)

check('05-p8', answer)

✅ 05-p8: correct! Nicely done.


True

### 🧩 Problem 5.9: Busy series

A *series* is one (crop, market) pair. How many series have an **average weekly volume above 25,000 kg**?

*Expected type:* integer.

In [10]:
s = prices.groupby(["crop", "market"])["volume_kg"].mean()
answer = int((s > 25000).sum())

check('05-p9', answer)

✅ 05-p9: correct! Nicely done.


True

### 🧩 Problem 5.10: Market league table (named aggregation)

Build a summary by `market` with named aggregation: `total_volume_t` = total `volume_kg` **in tonnes** (÷1000). Return the total volume in tonnes for **Wakulima Market**, rounded to 1 decimal.

*Expected type:* float.

In [11]:
s = prices.groupby("market").agg(total_volume_t=("volume_kg", lambda s: s.sum() / 1000))
answer = round(float(s.loc["Wakulima Market", "total_volume_t"]), 1)

check('05-p10', answer)

✅ 05-p10: correct! Nicely done.


True